# 🚀 MarketingFlow AI — Multi-Agent Marketing Agency System

**Architecture:** LangGraph + Google Gemini (gemini-2.5-flash) + Google Search Grounding + Semantic RAG

---

### 📌 System Overview
MarketingFlow AI is an autonomous multi-agent agency that receives client business details, analyzes the request, retrieves grounded context, conducts live market research via Google Search, drafts full marketing strategy, produces ready-to-publish social media content, reviews quality against brand guidelines, and compiles the final deliverable.

## 1. Installation & Environment Setup
Run the cell below to install required packages if running in Google Colab or a new environment.

In [ ]:
# Install dependencies
!pip install -q langgraph langchain-core langchain-google-genai python-dotenv pydantic

In [ ]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv(override=True)

# Set Gemini API Key if not already in environment
if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google Gemini API Key: ")

os.environ["GEMINI_MODEL"] = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")
os.environ["GEMINI_EMBEDDING_MODEL"] = os.getenv("GEMINI_EMBEDDING_MODEL", "models/gemini-embedding-001")
print("✅ Environment configured successfully with model:", os.environ["GEMINI_MODEL"])

## 2. Define State Schema
The  dictionary tracks all workflow variables across agent nodes.

In [ ]:
from typing import Optional, TypedDict

class MarketingState(TypedDict, total=False):
    client: dict
    request: str
    duration_days: int
    route: str
    route_reason: str
    knowledge_context: str
    research: str
    strategy: str
    content_plan: str
    review: str
    final_output: str

print("✅ MarketingState schema defined.")

## 3. Configure LLM & Embeddings
Setting up Google Generative AI chat model and vector embeddings.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

def get_llm(temperature: float = 0.3) -> ChatGoogleGenerativeAI:
    return ChatGoogleGenerativeAI(
        model=os.getenv("GEMINI_MODEL", "gemini-2.5-flash"),
        temperature=temperature,
        thinking_budget=0,
    )

def get_embeddings() -> GoogleGenerativeAIEmbeddings:
    return GoogleGenerativeAIEmbeddings(
        model=os.getenv("GEMINI_EMBEDDING_MODEL", "models/gemini-embedding-001")
    )

# Quick verification
test_resp = get_llm().invoke("Ping: Confirm system online.")
print("LLM Status:", test_resp.content)

## 4. Multi-Agent Nodes Implementation
Here we implement the specialized agents: Supervisor, RAG Knowledge, Web Research, Marketing Strategist, Content Creator, Reviewer, and Finalizer.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore

class RouteDecision(BaseModel):
    route: Literal["knowledge_answer", "content_only", "strategy_only", "full_campaign"]
    reason: str = Field(description="Brief explanation for the routing decision")

def client_to_text(client: dict) -> str:
    return "
".join(f"- {k}: {v}" for k, v in client.items() if v)

def message_to_text(response) -> str:
    if isinstance(response, str):
        return response
    if hasattr(response, "content"):
        if isinstance(response.content, str):
            return response.content
        return str(response.content)
    return str(response)

print("✅ Helper utilities ready.")

In [ ]:
# 1. Supervisor Agent
def supervisor_agent(state: MarketingState) -> dict:
    client = state.get("client", {})
    request = state.get("request", "")
    llm = get_llm(temperature=0)
    router = llm.with_structured_output(RouteDecision)
    prompt = f"""You are the Supervisor Agent for an AI marketing agency.
Choose exactly one route:
- knowledge_answer: factual question about the client's business/offers.
- content_only: user wants quick copy/captions without market research.
- strategy_only: user wants marketing strategy only without full calendar.
- full_campaign: full content calendar/campaign requiring RAG + web research + strategy + content.

Client:
{client_to_text(client)}

User Request: {request}
"""
    try:
        decision = router.invoke(prompt)
        return {"route": decision.route, "route_reason": decision.reason}
    except Exception as exc:
        return {"route": "full_campaign", "route_reason": f"Fallback: {exc}"}

# 2. Client Knowledge RAG Agent
def client_knowledge_agent(state: MarketingState) -> dict:
    client = state.get("client", {})
    request = state.get("request", "")
    chunks = [f"{k.replace('_', ' ').title()}: {v}" for k, v in client.items() if v]
    if not chunks:
        return {"knowledge_context": "No client knowledge provided."}
    docs = [Document(page_content=c) for c in chunks]
    try:
        vs = InMemoryVectorStore(embedding=get_embeddings())
        vs.add_documents(docs)
        results = vs.similarity_search(request or "business overview", k=min(6, len(docs)))
        return {"knowledge_context": "
".join(f"- {d.page_content}" for d in results)}
    except Exception as exc:
        return {"knowledge_context": "
".join(f"- {d.page_content}" for d in docs[:6])}

# 3. Web Research Agent (Google Search Grounding)
def research_agent(state: MarketingState) -> dict:
    client = state.get("client", {})
    request = state.get("request", "")
    knowledge = state.get("knowledge_context", "")
    model = get_llm(temperature=0.2)
    prompt = f"""You are a marketing research agent. Research fresh, useful market trends, audience pain points, and competitors.
Client: {client_to_text(client)}
Retrieved Facts: {knowledge}
Task: {request}
"""
    try:
        model_search = model.bind_tools([{"google_search": {}}])
        resp = model_search.invoke(prompt)
        return {"research": message_to_text(resp)}
    except Exception:
        resp = model.invoke(prompt)
        return {"research": message_to_text(resp)}

# 4. Strategy Agent
def strategy_agent(state: MarketingState) -> dict:
    llm = get_llm(temperature=0.4)
    duration = state.get("duration_days", 7)
    prompt = f"""You are a senior marketing strategist. Build a practical {duration}-day marketing strategy.
Include: 1. Brand Diagnosis, 2. Target Audience, 3. 3-5 Content Pillars, 4. Positioning, 5. Funnel & CTA strategy.
Client Facts: {state.get('knowledge_context')}
Market Research: {state.get('research')}
Task: {state.get('request')}
"""
    resp = llm.invoke(prompt)
    return {"strategy": message_to_text(resp)}

# 5. Content Creator Agent
def content_creator_agent(state: MarketingState) -> dict:
    llm = get_llm(temperature=0.65)
    duration = state.get("duration_days", 7)
    prompt = f"""You are a marketing content creator. Produce ready-to-use content for {duration} days.
For each day include: Day number, Platform, Format (reel/carousel/post), Objective, Hook, Full Caption/Script, CTA, Hashtags.
Client Facts: {state.get('knowledge_context')}
Strategy: {state.get('strategy')}
Task: {state.get('request')}
"""
    resp = llm.invoke(prompt)
    return {"content_plan": message_to_text(resp)}

# 6. Quality Reviewer Agent
def reviewer_agent(state: MarketingState) -> dict:
    llm = get_llm(temperature=0.1)
    prompt = f"""Review the content plan for factual consistency, brand safety, tone, and CTA clarity.
Return: PASS/NEEDS_CHANGES + Short Quality Notes + Approved Content.
Draft: {state.get('content_plan')}
"""
    resp = llm.invoke(prompt)
    return {"review": message_to_text(resp)}

# 7. Finalizer Agent
def finalizer_agent(state: MarketingState) -> dict:
    final_out = f"""# MarketingFlow AI Deliverable

## Strategy
{state.get('strategy', 'Not requested.')}

## Research Insights
{state.get('research', 'Not requested.')}

## Content Plan
{state.get('content_plan', 'Not requested.')}

## Quality Review
{state.get('review', 'Not requested.')}
""".strip()
    return {"final_output": final_out}

def knowledge_answer_agent(state: MarketingState) -> dict:
    llm = get_llm(temperature=0.1)
    resp = llm.invoke(f"Answer using client context only: {state.get('knowledge_context')}
Question: {state.get('request')}")
    return {"final_output": message_to_text(resp)}

print("✅ All Agent Nodes implemented successfully.")

## 5. Building the LangGraph Multi-Agent Workflow
Here we connect nodes with conditional routing edges into an executable state graph.

In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(MarketingState)

# Add all nodes
builder.add_node("supervisor", supervisor_agent)
builder.add_node("client_knowledge", client_knowledge_agent)
builder.add_node("research", research_agent)
builder.add_node("strategy", strategy_agent)
builder.add_node("content_creator", content_creator_agent)
builder.add_node("reviewer", reviewer_agent)
builder.add_node("knowledge_answer", knowledge_answer_agent)
builder.add_node("finalizer", finalizer_agent)

# Conditional routing functions
def after_supervisor(state: MarketingState) -> str:
    return state.get("route", "full_campaign")

def after_knowledge(state: MarketingState) -> str:
    r = state.get("route", "full_campaign")
    if r == "knowledge_answer": return "knowledge_answer"
    if r == "content_only": return "content_creator"
    return "research"

def after_strategy(state: MarketingState) -> str:
    return "finalizer" if state.get("route") == "strategy_only" else "content_creator"

# Add edges
builder.add_edge(START, "supervisor")
builder.add_conditional_edges("supervisor", after_supervisor, {
    "knowledge_answer": "client_knowledge",
    "content_only": "client_knowledge",
    "strategy_only": "client_knowledge",
    "full_campaign": "client_knowledge",
})
builder.add_conditional_edges("client_knowledge", after_knowledge, {
    "knowledge_answer": "knowledge_answer",
    "content_creator": "content_creator",
    "research": "research",
})
builder.add_edge("knowledge_answer", END)
builder.add_edge("research", "strategy")
builder.add_conditional_edges("strategy", after_strategy, {
    "finalizer": "finalizer",
    "content_creator": "content_creator",
})
builder.add_edge("content_creator", "reviewer")
builder.add_edge("reviewer", "finalizer")
builder.add_edge("finalizer", END)

graph = builder.compile()
print("✅ LangGraph Multi-Agent compiled successfully!")

## 6. Live Execution Demo
Run a complete 7-day marketing campaign generation pipeline through the graph.

In [ ]:
demo_input = {
    "client": {
        "business_name": "FitZone",
        "industry": "Fitness & Health",
        "products_services": "Personalized Online Fitness & Nutrition Coaching",
        "target_audience": "Busy professionals aged 25-40 with limited time",
        "goal": "Generate qualified consultation leads",
        "platforms": "Instagram, LinkedIn, Facebook",
        "tone": "Inspiring, practical, and empathetic",
        "special_offers": "Free 20-minute Strategy Session",
        "extra_knowledge": "No fad diets or extreme workouts. We focus on 1% daily sustainable habits.",
    },
    "request": "Build a complete 7-day marketing campaign with full ready-to-post content.",
    "duration_days": 7,
}

print("🚀 Invoking LangGraph Multi-Agent Workflow...")
result = graph.invoke(demo_input)
print(f"
🎯 Route Selected: {result.get('route')} ({result.get('route_reason')})")
print("="*70)
print(result.get("final_output"))